# Dataspace Automated BFF Contract Negotiation

This notebook demonstrates the high-level Backend-For-Frontend (BFF) RPC workflow for automated contract negotiation:
1. Executes prerequisites (`setup_catalog_policies.ipynb`) if catalog and policy targets are not already loaded.
2. Automated Contract Negotiation from Consumer Request (`/bff-rpc/setup-request-init` and `/bff-rpc/setup-agreement`).
3. Automated Contract Negotiation from Provider Offer (`/bff-rpc/setup-offer-init`, `/bff-rpc/setup-acceptance`, and `/bff-rpc/setup-agreement`).
4. Extracts and displays the finalized contract agreement and `agreement_id`.

## 1. Prerequisites
Ensure that catalog and policies are populated and target identifiers are available. If not already loaded in the environment, `setup_catalog_policies.ipynb` is automatically executed.

In [ ]:
if "target_id" not in globals() or not target_id:
    %run ./setup_catalog_policies.ipynb

# Part I: Automated Contract Negotiation — Consumer Request Flow

The BFF RPC layer simplifies negotiation by collapsing multi-step exchanges into coordinated high-level calls.
In this flow, the Consumer initiates the negotiation request, and the Provider automatically agrees, verifies, and finalizes the contract.

## 2. Consumer Negotiation Request Initiation (`/bff-rpc/setup-request-init`)
The Consumer submits a contract request referencing the target dataset and offer policy.
The Provider records the request in `REQUESTED` state and returns `providerPid`.

In [ ]:
url = data_space_consumer + "/dsp/current/negotiations/bff-rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [{"action": "use"}]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
cn_consumer_id = response_as_json["response"]["consumerPid"]
cn_provider_id = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

## 3. Provider Auto-Agreement & Finalization (`/bff-rpc/setup-agreement`)
The Provider executes agreement, verification, and finalization in a single automated BFF call, transitioning the negotiation to `FINALIZED`.

In [ ]:
url = data_space_provider + "/dsp/current/negotiations/bff-rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

## 4. Final Agreement Details
Display the finalized contract agreement and save the `agreement_id` for transfer execution.

In [ ]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

# Part II: Automated Contract Negotiation — Provider Offer Flow

In this alternative flow, the negotiation begins with an initial offer dispatched by the Provider.
The Consumer accepts the offer, and the Provider then completes agreement and finalization.

## 5. Provider Offer Initiation (`/bff-rpc/setup-offer-init`)
The Provider initiates negotiation by dispatching an offer directly to the Consumer via the BFF offer initiation endpoint.

In [ ]:
url = data_space_provider + "/dsp/current/negotiations/bff-rpc/setup-offer-init"
payload = {
    "associatedAgentPeer": data_space_consumer_participant_id,
    "providerAddress": data_space_consumer_dsp_endpoint,
    "callbackAddress": data_space_provider_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [{"action": "use"}]
    }
}

try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

## 6. Consumer Acceptance (`/bff-rpc/setup-acceptance`)
The Consumer accepts the Provider's offer via the BFF acceptance endpoint.
The negotiation transitions to `ACCEPTED`.

In [ ]:
url = data_space_consumer + "/dsp/current/negotiations/bff-rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
    agreement = response_as_json
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

## 7. Provider Auto-Agreement & Finalization (`/bff-rpc/setup-agreement`)
The Provider completes agreement exchange and finalizes the negotiation.

In [ ]:
url = data_space_provider + "/dsp/current/negotiations/bff-rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

## 8. Final Agreement Details
Display the finalized contract agreement and verify the generated `agreement_id`.

In [ ]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))